# GökTürk-Ultra — Kaggle eğitim denemesi (2×T4)

Bu notebook **altyapıyı** çalıştırır ve doğrular; eğitilmiş model teslim etmez.

1. Ortamı ve donanımı kontrol eder (GPU, CUDA, VRAM, paket sürümleri).
2. Paketleri kurar — 4-bit gpt-oss için **Unsloth** dahil.
3. Veri dosyasını şemaya ve kalite kurallarına göre doğrular.
4. **Duman testi**: rastgele küçük modelle 2 adım. Yalnızca boru hattını kanıtlar.
5. İki eğitim arka ucunu (generic transformers / Unsloth 4-bit) ön denetimden geçirir.
6. Uygun olan yolla **kısa ölçüm koşusu** (60 adım) yapar; uygun değilse dürüstçe durur.
7. Çıktıları `/kaggle/working` altında kaydeder.
8. *(İsteğe bağlı)* Adaptörü Hugging Face'e yükler; GGUF'un bu makinede
   mümkün olup olmadığını sayılarla söyler (dönüşümü çalıştırmaz).

## Kaggle'da yapılacaklar

* **Accelerator → GPU T4 x2**, **Internet → On**.
* **Add-ons → Secrets**: yalnızca `EVREN_API_KEY` (adres ve model kimlikleri `configs/teachers.yaml` içinde doğrulanmış olarak hazır).
* Eğitim verisini (JSONL) **Kaggle Dataset** olarak ekleyin; notebook `/kaggle/input` altında arar.

## Neden Unsloth?

`openai/gpt-oss-20b` ağırlıkları MXFP4 olarak yayınlanmıştır ve generic
transformers+bitsandbytes yoluyla 16 GB'lık bir karta sığmaz (bf16 LoRA ≈44 GB).
Unsloth'un 4-bit QLoRA çekirdekleri 20B modeli ≈14 GB VRAM'de eğitir. Bu yüzden
bu notebook Unsloth yolunu birinci seçenek olarak dener.

**Uyarı:** T4 bfloat16 desteklemez; Unsloth gpt-oss'ta MoE katmanlarını
float32'ye taşıyarak taşmayı engeller, ancak uzun koşularda kayıp ıraksaması
geçmişte raporlanmıştır. Kayıp eğrisini izleyin; 60 adımlık ölçüm koşusu bunun
için var.

> **Ön koşullar:** Notebook ayarlarında **Accelerator: GPU T4 ×2** ve **Internet: On**.
> Depo, PR dalından klonlanır (`REPO_BRANCH`); dal main'e birleşince `main` yapılabilir.
> Tek kart kullanılır (`CUDA_VISIBLE_DEVICES=0`); ikinci T4 bilinçli olarak boş bırakılır.


## 1) Ortam bilgisi

In [ ]:
import os, platform, subprocess, sys

# Kaggle iki T4 gösterir; bu repo model paralelliği UYGULAMAZ ve 4-bit
# ağırlıklar DataParallel ile çoğaltılamaz. Tek kartı, torch import'undan
# önce sabitliyoruz ("Caught RuntimeError in replica 0" hatasının kökü).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"], "(tek kart)")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Çalışma dizini:", os.getcwd())
print("CPU çekirdeği:", os.cpu_count())
try:
    with open("/proc/meminfo") as fh:
        for line in fh:
            if line.startswith("MemTotal"):
                print("RAM:", round(int(line.split()[1]) / 1024**2, 1), "GB")
                break
except OSError:
    pass

print("\n--- nvidia-smi ---")
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(r.stdout.strip() or r.stderr.strip() or "nvidia-smi bulunamadı")

## 2) Paketler

Unsloth kurulumu kendi `torch`/`triton` sürümlerini isteyebilir; kurulumdan
sonra **Runtime → Restart** gerekirse notebook'u baştan çalıştırın.
`INSTALL=False` ile Kaggle imajındaki mevcut sürümleri kullanabilirsiniz.

In [ ]:
INSTALL = True  # atlamak için False yapın
PINNED = [
    "transformers>=4.55,<6",   # gpt-oss mimarisi
    "peft>=0.12",
    "accelerate>=0.33",
    "pyyaml>=6.0",
]
UNSLOTH_SPEC = "unsloth"      # 4-bit QLoRA yolu (≈14 GB VRAM)

if INSTALL:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=False)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", UNSLOTH_SPEC, "xformers", "trl"],
        check=False,
    )

import importlib
for name in ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "unsloth", "yaml"]:
    try:
        mod = importlib.import_module(name)
        print(f"{name:<14}", getattr(mod, "__version__", "?"))
    except Exception as exc:
        print(f"{name:<14} KURULU DEĞİL ({type(exc).__name__})")

## 3) Depoyu hazırla ve Secrets'ı ortama aktar

Secrets yalnızca ortam değişkenine aktarılır; anahtar hiçbir hücrede
yazdırılmaz veya dosyaya yazılmaz.

In [ ]:
import os, pathlib, subprocess, sys

# ---------------------------------------------------------------------------
# KOD TAZELİĞİ — bu hücre her koşuda depoyu günceller.
# Kaggle'da /kaggle/working kalıcı olmadığı için (veya eski bir klon kaldığı
# için) bayat kod çalıştırmak kolaydır: 2026-10-25'te tam olarak bu oldu,
# `gokturk.lora_gguf` import edilemedi. Artık güncelleme ZORUNLU (reset --hard).
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/a23521745-hub/GokTurk-Ultra.git"  # gerekirse değiştirin
REPO_BRANCH = "main"        # birleşmemiş bir PR'ı denemek için dal adı yazılabilir
REPO_DIR = pathlib.Path("/kaggle/working/GokTurk-Ultra")


def _git(*args, check=False):
    return subprocess.run(["git", *args], capture_output=True, text=True, check=check)


def _refresh_repo() -> None:
    """Depoyu REPO_BRANCH'in en son commit'ine getir. Yerel değişiklikler gider."""
    if not REPO_DIR.exists():
        cloned = _git("clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR))
        if cloned.returncode != 0:
            print("UYARI: dal klonlanamadı, main çekiliyor:", REPO_BRANCH)
            _git("clone", "--depth", "1", REPO_URL, str(REPO_DIR), check=True)
        return
    print("Depo zaten var, en son commit'e getiriliyor:", REPO_DIR)
    fetched = _git("-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH)
    if fetched.returncode != 0:
        print("UYARI: dal getirilemedi, mevcut kod kullanılacak:", REPO_BRANCH)
        return
    _git("-C", str(REPO_DIR), "checkout", "-q", "-B", REPO_BRANCH, "FETCH_HEAD")
    _git("-C", str(REPO_DIR), "reset", "-q", "--hard", "FETCH_HEAD")


_refresh_repo()

_branch = _git("-C", str(REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD").stdout.strip()
_commit = _git("-C", str(REPO_DIR), "rev-parse", "--short", "HEAD").stdout.strip()
_subject = _git("-C", str(REPO_DIR), "log", "-1", "--pretty=%s").stdout.strip()
print("Çalışılan dal:", _branch, "| commit:", _commit)
print("Son commit   :", _subject[:100])

# Hangi yetenekler bu kodda var? (bayat klon sessizce yanlış davranmasın)
_CAPABILITIES = {
    "configs/training_qwen3_5_4b_t4.yaml": "4B Kaggle profili",
    "src/gokturk/lora_gguf.py": "adaptör-only GGUF planı",
    "src/gokturk/serve.py": "servis planı",
}
_missing = []
for relative, label in _CAPABILITIES.items():
    if not (REPO_DIR / relative).exists():
        _missing.append(f"{label} ({relative})")
if _missing:
    print("UYARI: bu kopyada eksik yetenekler:", "; ".join(_missing))
    print("       Depo bayat olabilir. Çözüm: bu hücreyi tekrar çalıştırın;")
    print("       olmazsa Runtime > Restart sonrası hücreleri baştan koşun.")
else:
    print("Yetkinlikler : 4B profil + adaptör-only GGUF + servis planı (hepsi mevcut)")

sys.path.insert(0, str(REPO_DIR / "src"))
# Aynı isimli eski bir kurulum önbellekte kalmasın (git pull sonrası tipik tuzak).
for _name in [m for m in list(sys.modules) if m == "gokturk" or m.startswith("gokturk.")]:
    del sys.modules[_name]
print("src yolu eklendi:", REPO_DIR / "src")

SECRET_KEYS = ["EVREN_API_KEY", "HF_TOKEN"]  # HF_TOKEN yalnızca yükleme için gerekir
# İsteğe bağlı geçersiz kılmalar (teachers.yaml içinde env: olarak kullanılırsa)
OPTIONAL_SECRET_KEYS = ["EVREN_BASE_URL", "EVREN_MODEL_MIMO", "EVREN_MODEL_GLM"]
try:
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
    for key in SECRET_KEYS + OPTIONAL_SECRET_KEYS:
        try:
            os.environ[key] = client.get_secret(key)
        except Exception:
            pass
except Exception:
    print("kaggle_secrets yok (Kaggle dışında çalışıyor olabilirsiniz)")

for key in SECRET_KEYS:
    print(f"{key:<18}", "TANIMLI" if os.environ.get(key) else "yok")

# İnternet kapalıysa HF indirmesi de EVREN de çalışmaz (Notebook: Settings > Internet > On).
_check = subprocess.run(
    [sys.executable, "-c", "import socket; socket.create_connection(('huggingface.co', 443), timeout=5); print('internet: AÇIK')"],
    capture_output=True, text=True,
)
print(_check.stdout.strip() or "internet: KAPALI -> Notebook ayarlarından Internet'i açın (model indirilemez)")


## 3b) Öğretmen model kimlikleri — tahmin yok, sağlayıcıya sorulur

`configs/teachers.yaml` içindeki `mimo-v2.6-pro` ve `glm-5.3` kimlikleri
EVREN'in herkese açık model listesinden alınmıştır. Bu hücre **çalışma anında**
sağlayıcının `GET /models` yanıtını çeker ve yapılandırmayla karşılaştırır:
kimlik değişmişse üretim başlamadan görürsünüz.

`EVREN_API_KEY` yoksa hücre hata vermez; ne yapılacağını yazar ve devam eder.
Eğitim (8. adım) bu hücreye bağlı değildir; kimlikler yalnızca **veri üretimi**
için gerekir.


In [ ]:
import json

from gokturk.config import ConfigError, load_teachers_config
from gokturk.teachers import TeacherError, TeacherPool

try:
    teachers_cfg = load_teachers_config(str(REPO_DIR / "configs" / "teachers.yaml"))
    model_report = TeacherPool(teachers_cfg).list_models()
    print("Sağlayıcının listelediği modeller:")
    for model_id in model_report["available"]:
        print("  -", model_id)
    print("\nYapılandırılmış öğretmenler:")
    print(json.dumps(model_report["configured"], ensure_ascii=False, indent=2))
    if model_report["ok"]:
        print("\n>>> KİMLİKLER DOĞRULANDI: her iki öğretmen de listede.")
    else:
        print("\n>>> EKSİK KİMLİK:", ", ".join(model_report["missing"]))
        print("    configs/teachers.yaml içindeki `model` alanlarını yukarıdaki listeden düzeltin.")
except ConfigError as exc:
    print("Yapılandırma okunamadı:", exc)
except TeacherError as exc:
    print("Model listesi alınamadı:", type(exc).__name__, exc)
    print("En sık sebep: EVREN_API_KEY tanımlı değil (Add-ons > Secrets > Attach).")


## 4) Donanım ön denetimi — iki arka uç karşılaştırması

Generic yol (`transformers`) ve Unsloth yolu ayrı ayrı değerlendirilir.
Hangisinin gerçekten çalışabileceğini bu hücre söyler.

In [ ]:
from gokturk.env_check import check_environment

# NOT: Bu hücre ESKİ gpt-oss-20b yolunu karşılaştırır — bilgi amaçlı.
# Gerçek koşunun profilini 8. hücre seçer (varsayılan: qwen3.5-4b, Unsloth'suz).
print('NOT: bu hücre gpt-oss yolu içindir; gerçek koşu profilini 8. hücre seçer.')

MODEL_GENERIC = "openai/gpt-oss-20b"
MODEL_UNSLOTH = "unsloth/gpt-oss-20b"

generic_report = check_environment(
    model_name=MODEL_GENERIC, method="qlora", dtype="auto", load_in_4bit=True, seq_len=1024
)
print("========== GENERIC (transformers + bitsandbytes) ==========")
print(generic_report.render()[-1200:])

try:
    from gokturk.unsloth_backend import detect_unsloth, preflight_unsloth

    unsloth_status = detect_unsloth()
    unsloth_report = preflight_unsloth(
        MODEL_UNSLOTH, load_in_4bit=True, hardware=generic_report.hardware, max_seq_length=1024
    )
    print("\n========== UNSLOTH (4-bit QLoRA) ==========")
    print(unsloth_report.render()[-1600:])
except Exception as exc:  # Unsloth yoksa da notebook devam etmeli
    print("\nUnsloth ön denetimi çalıştırılamadı:", type(exc).__name__, exc)
    unsloth_report = None

print("\n>>> GENERIC uygun mu:", "EVET" if generic_report.ok else "HAYIR")
print(">>> UNSLOTH uygun mu:", "EVET" if (unsloth_report and unsloth_report.ok) else "HAYIR")

## 5) Eğitim verisini doğrula

Veri dosyasını `/kaggle/input` altında arar. Şema: `id`, `prompt`, `response`,
`source` (`src/gokturk/schema.py`). Bozuk satırlar, tekrarlar ve aşırı uzun
kayıtlar raporlanır; yalnızca temizlenmiş veri `data/processed` altına yazılır.

In [ ]:
import json, pathlib

from gokturk.data_quality import build_datasets, read_jsonl

input_root = pathlib.Path("/kaggle/input")
candidates = sorted(input_root.rglob("*.jsonl")) if input_root.exists() else []
print("Bulunan JSONL dosyaları:")
for path in candidates[:20]:
    print("  -", path)

DATA_FILE = None  # örn. pathlib.Path("/kaggle/input/gtu-data/pro.jsonl")

if DATA_FILE is None:
    preferred = [p for p in candidates if "train" in p.name.lower() or "pro" in p.name.lower()]
    DATA_FILE = (preferred or candidates or [REPO_DIR / "data" / "samples" / "sample_train.jsonl"])[0]

print("\nKullanılacak veri:", DATA_FILE)
records, issues = read_jsonl(DATA_FILE)
print(f"Okunan kayıt: {len(records)} | Ayrıştırma hatası: {len(issues)}")

PROCESSED = pathlib.Path("/kaggle/working/data/processed")
if records:
    try:
        data_report = build_datasets(DATA_FILE, PROCESSED)
        print("\nVeri raporu:", data_report.summary())
    except ValueError as exc:
        print("Veri kalitesi hatası:", exc)
else:
    print("UYARI: veri dosyası boş.")

## 6) Sentetik veri planı (ağa çıkmaz)

Üretim modülü 10 görev ailesi (kodlama, hata ayıklama, akıl yürütme, ileri
Türkçe, sistem mimarisi, araç çağırma, talimat takibi, özetleme, yeniden yazma,
çeviri) üzerinden deterministik bir plan ve kaba maliyet tahmini üretir.
Bu hücre **API çağrısı yapmaz**.

In [ ]:
from gokturk.data_generation import plan_only

plan_summary = plan_only(
    target=1000,
    seed=0,
    plan_path="/kaggle/working/plan_1000.json",
)
print("Aile dağılımı:", json.dumps(plan_summary["counts"]["family"], ensure_ascii=False, indent=2))
print("\nSeviye dağılımı:", plan_summary["counts"]["level"])
print("Maliyet tahmini:", json.dumps(plan_summary["cost"], ensure_ascii=False, indent=2))
print("\nNot: gerçek üretim için `python -m gokturk.data_generation --mode pro ...` gerekir.")
print("Model kimlikleri configs/teachers.yaml içinde hazır; tek zorunlu secret EVREN_API_KEY.")

## 7) Duman testi (indirme gerektirmez)

Rastgele başlatılmış küçük model + dosyadan üretilen kelime düzeyi tokenizer ile
2 optimizasyon adımı. Amaç: veri yükleme, prompt maskeleme, LoRA, Trainer ve
kaydetme yolunun uçtan uca çalıştığını göstermek.

**Bu bir eğitim değildir ve hiçbir kalite sonucu üretmez.**

In [ ]:
import gokturk.train as gtu_train
from gokturk.config import load_training_config

SMOKE_OUT = "/kaggle/working/smoke"
cfg = load_training_config(REPO_DIR / "configs" / "training.yaml")
if DATA_FILE.exists():
    cfg.data.train_file = str(DATA_FILE)
    cfg.data.eval_file = str(DATA_FILE)

smoke_code = gtu_train.run_training(cfg, smoke=True, max_steps=2, output_dir=SMOKE_OUT)
print("\nDuman testi çıkış kodu:", smoke_code, "(0 = başarılı)")

manifest_path = pathlib.Path(SMOKE_OUT) / "run_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("kind:", manifest["kind"], "| örnek:", manifest["data"]["train_examples"])
    print("not:", manifest["note"])
else:
    print("MANİFEST YOK: duman testi tamamlanmadı.")

## 8) Gerçek eğitim: 60 adımlık ölçüm koşusu

Profil **açıkça** seçilir; varsayılan `qwen3.5-4b` (transformers + bitsandbytes
4-bit QLoRA). Unsloth yolu (`gpt-oss-20b-unsloth`) yalnızca 20B MoE içindir ve
füzyonlu kayıp (fused loss) çekirdeği yüzünden kırılgandır: 2026-10-25'te
Kaggle'da `grad_and_value ... Expected f(*args) to return a scalar Tensor`
hatasıyla **ilk adımda** düştü. 4B'de Unsloth gerekmez.

`max_steps=60` bilinçlidir: kayıp eğrisinin ilk sinyalini 12 saatlik oturumu
tüketmeden görürsünüz. Kararlıysa adım sayısını artırın.

Disk, VRAM'den önce gelir (Kaggle'da boş alan ~19 GB): ön denetim
`weights-disk-insufficient` derse eğitim **başlamaz**.


In [ ]:
from gokturk.config import load_training_config
from gokturk.env_check import check_environment

# İlk gerçek koşu: Qwen3.5-4B, transformers + bitsandbytes (4-bit QLoRA).
# Alternatif: "gpt-oss-20b-unsloth" — 20B MoE içindir; fused-loss kırılganlığı var.
RUN_PROFILE = "qwen3.5-4b"

PROFILES = {
    "qwen3.5-4b": (REPO_DIR / "configs" / "training_qwen3_5_4b_t4.yaml", "transformers"),
    "gpt-oss-20b-unsloth": (REPO_DIR / "configs" / "training_unsloth_t4.yaml", "unsloth"),
}
TRAIN_OUT = "/kaggle/working/gokturk-run"

config_path, BACKEND = PROFILES[RUN_PROFILE]

if not config_path.is_file():
    print("HATA: config bulunamadı:", config_path)
    print("Sebep: depo yanlış dalda (main'de yeni profiller yok) — 3. hücreye bakın.")
    print("Çözüm: 3. hücrede REPO_BRANCH doğru mu, Runtime > Restart, hücreleri baştan çalıştırın.")
    print("Mevcut profiller:", sorted(p.name for p in (REPO_DIR / "configs").glob("training*.yaml")))
    raise SystemExit(f"config yok: {config_path.name}")

run_cfg = load_training_config(config_path)
print("Profil:", RUN_PROFILE, "| config:", config_path.name, "| arka uç:", BACKEND)
print("Taban model:", run_cfg.model.name_or_path,
      "| 4-bit:", run_cfg.model.load_in_4bit,
      "| seq:", run_cfg.data.max_seq_len)

pre = check_environment(
    model_name=run_cfg.model.name_or_path,
    method=run_cfg.optimization.method,
    dtype=run_cfg.model.dtype,
    load_in_4bit=run_cfg.model.load_in_4bit,
    seq_len=run_cfg.data.max_seq_len,
    batch_size=run_cfg.optimization.per_device_train_batch_size,
    check_hub=False,
)
print(pre.render()[-1600:])
print(">>> Ön denetim:", "GEÇTİ" if pre.ok else "ENGEL VAR")

if not pre.ok:
    print("\nEğitim atlandı. Engelleyici bulgular:")
    for finding in pre.blocking:
        print(f"  [{finding.code}] {finding.message}")
    print("\nSeçenekler:")
    print("  1) Daha küçük taban (4B) veya daha büyük disk")
    print("  2) max_seq_len / batch düşürüp tekrar --plan-only")
else:
    train_file = PROCESSED / "train.jsonl"
    valid_file = PROCESSED / "valid.jsonl"
    run_cfg.data.train_file = str(train_file) if train_file.exists() else str(DATA_FILE)
    run_cfg.data.eval_file = str(valid_file) if valid_file.exists() else str(DATA_FILE)
    # KÜÇÜK VERİ UYARISI: 4 örnekle 60 adım = 60 epoch = ezberleme.
    # train_loss düşerken eval_loss yüksek kalır; bu "öğrendi" demek değildir.
    import pathlib as _pl

    _train_rows = sum(1 for line in open(run_cfg.data.train_file, encoding="utf-8") if line.strip())
    run_cfg.optimization.max_steps = 60
    if _train_rows < 100:
        print("\n" + "!" * 70)
        print(f"UYARI: eğitim seti yalnızca {_train_rows} örnek.")
        print("60 adım bu veriyle ~%d epoch demek: model 4 örneği EZBERLER." % round(60 * run_cfg.optimization.per_device_train_batch_size * run_cfg.optimization.gradient_accumulation_steps / max(_train_rows, 1)))
        print("Bu koşu yalnızca boru hattını kanıtlar; adaptör yayınlanabilir DEĞİL.")
        print("Gerçek koşu: data/processed/train.jsonl (>=9000 örnek) + max_steps 600-1200.")
        print("!" * 70 + "\n")
    code = gtu_train.run_training(run_cfg, output_dir=TRAIN_OUT, backend=BACKEND)
    print("Eğitim çıkış kodu:", code)
    if code != 0:
        print("Eğitim tamamlanmadı. Yukarıdaki tanılamayı inceleyin.")
        print("Yarıda kalan çıktılar (varsa):", TRAIN_OUT)


## 9) Kayıp eğrisi ve çıktı kontrolü

Kısa koşunun kayıp değerleri manifestten okunur. **Sayılar tek başına model
kalitesi göstergesi değildir**; yalnızca eğitimin çalıştığını ve ıraksamadığını
gösterir. Irksama (kayıp artışı / NaN) varsa koşu durdurulmalıdır.

In [ ]:
run_manifest = pathlib.Path(TRAIN_OUT) / "run_manifest.json"
if run_manifest.exists():
    m = json.loads(run_manifest.read_text())
    print("backend:", m.get("backend"), "| dtype:", m.get("resolved_dtype"))
    print("süre (sn):", m.get("duration_s"))
    print("örnek sayısı:", m.get("data"))
    print("metrikler:", json.dumps(m.get("metrics", {}), ensure_ascii=False, indent=2)[:800])
    adapter_dir = pathlib.Path(TRAIN_OUT) / "adapter"
    if adapter_dir.exists():
        files = sorted(p.name for p in adapter_dir.iterdir())
        print("\nadapter dosyaları:", files[:12])
    # Mekanik sağlık: NaN/ıraksama/ezberleme sinyali (kalite ölçmez)
    try:
        from gokturk.logcheck import analyze_manifest, render as render_logcheck

        print("\n" + render_logcheck(analyze_manifest(m, source=str(run_manifest))))
    except Exception as exc:  # tanılama aracı notebook'u düşürmesin
        print("logcheck çalıştırılamadı:", type(exc).__name__, exc)
else:
    print("Manifest yok: gerçek eğitim çalışmadı (yukarıdaki ön denetim sonucuna bakın).")

## 10) Değerlendirme çerçevesi

Bu hücre yalnızca **veri biçimi** kontrolü ve değerlendirme seti envanteri
yapar. Model çıktıları üzerinden skorlama, tahmin dosyası gerektirir ve
`python -m gokturk.evaluate --mode score --predictions ...` ile ayrı çalışır.

In [ ]:
from gokturk.evaluate import format_check, load_eval_set

eval_items = load_eval_set(REPO_DIR / "data" / "eval" / "turkish_eval.jsonl")
print(f"Değerlendirme seti: {len(eval_items)} öğe")
for category in sorted({item.category for item in eval_items}):
    n = sum(1 for item in eval_items if item.category == category)
    print(f"  {category:<24} {n}")

if (PROCESSED / "train.jsonl").exists():
    fmt = format_check(PROCESSED / "train.jsonl")
    print("\nBiçim kontrolü (model kalitesi ölçmez):")
    print(json.dumps({k: fmt[k] for k in ("records", "records_with_problems", "acceptable_ratio")},
                     ensure_ascii=False, indent=2))

## 11) (İsteğe bağlı) Adaptörü Hugging Face'e yükle

Varsayılan olarak **kapalıdır**: aşağıdaki hücre `RUN_PUBLISH = False` iken
hiçbir şey yapmaz. Açmak için:

1. **Add-ons → Secrets**'a `HF_TOKEN` ekleyin (yazma izinli token) ve **Attach**.
2. Hücrede `RUN_PUBLISH = True` ve `HF_REPO_ID = "kullanici/depo-adi"` yapın.

Yükleme yalnızca **adaptörü** gönderir (~yüz MB), 42 GB'lık birleşik modeli
değil. Model kartı `run_manifest.json`'dan üretilir; ölçülmemiş başarı iddiası
içermez. Depo varsayılan olarak **özel** açılır.

GGUF burada üretilmez: LoRA birleştirmesi ≈42 GB bellek ister, Kaggle'da
(29 GB RAM / 20 GB disk) mümkün değildir. Aşağıdaki hücre yalnızca **planı**
yazdırır.

Bu hücrenin tamamı **isteğe bağlıdır** ve her blok kendi hatasını yakalar:
bir import hatası eğitim çıktılarını çöpe atmaz. GGUF planı birleştirme
**yapmaz**; adaptör tek başına `.gguf`a çevrilebilir (≈2 GB RAM, 5–13 dk).


In [ ]:
import os, json, pathlib

# ---------------------------------------------------------------------------
# KURAL: bu hücre İSTEĞE BAĞLI işler yapar. İsteğe bağlı bir adım yüzünden
# 10 dakikalık eğitimin çıktısı çöpe gitmemeli — her blok kendi hatasını yakalar.
# (2026-10-25: `gokturk.lora_gguf` yok diye notebook burada düştü ve artefakt
#  zip'i hiç üretilmedi.)
# ---------------------------------------------------------------------------
RUN_PUBLISH = False                 # True yapmadan hiçbir şey yüklenmez
HF_REPO_ID = ""                     # ör: "kullanici/gokturk-ultra-lora"
PUBLIC = False                      # False -> özel depo

ADAPTER_DIR = str(pathlib.Path(TRAIN_OUT) / "adapter")
MANIFEST = str(pathlib.Path(TRAIN_OUT) / "run_manifest.json")
BASE_MODEL = run_cfg.model.name_or_path

print("HF_TOKEN:", "TANIMLI" if os.environ.get("HF_TOKEN") else "YOK (3. hücre secret listesini kontrol edin)")

# --- 1) Hugging Face yüklemesi (isteğe bağlı) --------------------------------
try:
    from gokturk.publish import PublishError, plan_push, push_adapter

    repo_ok = bool(HF_REPO_ID) and "/" in HF_REPO_ID
    if RUN_PUBLISH and not repo_ok:
        print("Atlandı: HF_REPO_ID boş ya da 'kullanici/depo-adi' biçiminde değil.")
    elif RUN_PUBLISH:
        manifest = json.loads(pathlib.Path(MANIFEST).read_text(encoding="utf-8"))
        result = push_adapter(ADAPTER_DIR, HF_REPO_ID, private=not PUBLIC, manifest=manifest)
        print("Yüklendi:", result["url"], "(özel)" if result["private"] else "(herkese açık)")
    else:
        plan = plan_push(ADAPTER_DIR, HF_REPO_ID or "kullanici/depo-adi")
        print("Kuru çalışma (hiçbir şey yüklenmedi): token var mı ->", plan.as_dict()["token_present"])
        print("Açmak için: RUN_PUBLISH = True, HF_REPO_ID = 'kullanici/depo-adi'")
except ImportError as exc:
    print("HF yükleme modülü yok:", exc, "-> 3. hücreyi tekrar çalıştırın (depo güncellensin)")
except Exception as exc:  # PublishError ve beklenmedik hatalar notebook'u düşürmesin
    print("Yükleme yapılmadı:", type(exc).__name__, exc)

# --- 2) GGUF: birleştirme YOK, adaptör tek başına ------------------------------
# plan_gguf (eski) BİRLEŞTİRME hesabı yapıyordu: ≈67 GB RAM => Kaggle'da HAYIR.
# Doğru yol: convert_lora_to_gguf.py adaptörü tek başına çevirir (≈2 GB RAM).
try:
    from gokturk.lora_gguf import plan_adapter_gguf, render_plan

    gguf = plan_adapter_gguf(
        ADAPTER_DIR,
        base_model=BASE_MODEL,
        out_dir="/kaggle/working/gguf",
        ram_gb=None,
        disk_gb=None,
    )
    print("\n" + render_plan(gguf))
except ImportError:
    print("\nAdaptör-only GGUF planı bu kod sürümünde yok (gokturk.lora_gguf).")
    print("Çözüm: 3. hücreyi tekrar çalıştırın (depo güncellenir), sonra bu hücreyi.")
    print("Elle: python -m gokturk.lora_gguf --adapter", ADAPTER_DIR, "--base-model", BASE_MODEL)
except Exception as exc:
    print("\nGGUF planı üretilemedi:", type(exc).__name__, exc)

print("\nNot: bu hücre isteğe bağlıdır; başarısız olsa da eğitim çıktıları korunur.")


## 12) Çıktıları kaydet

`/kaggle/working` 20 GB ile sınırlıdır ve oturum kapanınca kalıcı olmaz.
Adaptör ağırlıklarını kalıcı hale getirmek için Kaggle Dataset/Model olarak
yayımlayın.

In [ ]:
import zipfile

archive = pathlib.Path("/kaggle/working/gokturk_artifacts.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in [pathlib.Path(SMOKE_OUT), pathlib.Path(TRAIN_OUT), PROCESSED]:
        if folder.exists():
            for file in folder.rglob("*"):
                if file.is_file() and "adapter" not in str(file):
                    zf.write(file, file.relative_to("/kaggle/working"))
print("Arşiv:", archive, f"({archive.stat().st_size} bayt)")

adapter_paths = sorted(pathlib.Path("/kaggle/working").rglob("adapter_model.safetensors"))
print("Adaptör ağırlıkları:", adapter_paths or "yok (eğitim çalışmadı veya kaydedilmedi)")

## Bilinen sınırlamalar

* Bu notebook **eğitilmiş bir model teslim etmez**; altyapıyı ve dürüst ön
  denetimi çalıştırır.
* Unsloth yolu yalnızca ≈14 GB VRAM gerektirir *iddiasını* Unsloth belgelerine
  dayandırır; bu repoda gerçek bir Kaggle koşusu doğrulanmamıştır.
* T4'te bf16 yoktur; uzun koşularda kayıp kararlılığı izlenmelidir.
* İkinci T4 kullanılmaz (model paralelliği yok).
* Değerlendirme seti küçük ve kural tabanlıdır; benchmark yerine geçmez.
* Sentetik veri öğretmen çıktısıdır: insan incelemesi olmadan "altın veri"
  sayılmamalıdır (`docs/architecture.md` §4).